# Phase 3 : Deep Learning - Réseau de Neurones Convolutif (CNN)
Dans ce notebook, nous implémentons un véritable modèle de Deep Learning adapté au Traitement du Langage Naturel (NLP) avec **PyTorch**.

Nous remplaçons le simple perceptron multicouche (MLP) par un **CNN 1D**. Ce réseau utilise une couche d'Embedding pour comprendre le sens des mots, et des filtres convolutifs pour repérer des expressions de 2, 3 ou 4 mots (N-grams) caractéristiques d'un avis positif ou négatif.

In [1]:
import pandas as pd
import numpy as np
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from collections import Counter

# Ajout du chemin pour importer les modules locaux
sys.path.append(os.path.abspath('..'))
from src.data_loader import load_yelp_sample

### 1. Préparation des données (Tokenization et Plongements)

In [2]:
# 1. Chargement des données
print("Chargement des données...")
print("Mon dossier actuel est :", os.getcwd())
df = load_yelp_sample('../data/raw/review.json', n_rows=50000)
df['label'] = (df['stars'] <= 2).astype(int) # 1: Négatif, 0: Positif

# 2. Tokenization basique (séparer par les espaces et mettre en minuscules)
texts = df['text'].str.lower().str.split().tolist()
labels = df['label'].tolist()

# 3. Création du vocabulaire (on garde les 10 000 mots les plus fréquents)
vocab_size = 10000
all_words = [word for text in texts for word in text]
word_counts = Counter(all_words)
vocab = {word: i+2 for i, (word, _) in enumerate(word_counts.most_common(vocab_size - 2))}
vocab['<PAD>'] = 0 # Padding (remplissage pour avoir la même taille)
vocab['<UNK>'] = 1 # Unknown (mots inconnus)

# 4. Conversion des textes en séquences d'entiers et Padding (longueur fixe de 100 mots)
max_len = 100
X_seq = []
for text in texts:
    seq = [vocab.get(word, vocab['<UNK>']) for word in text]
    if len(seq) < max_len:
    # On ajoute des 0 à la fin pour les phrases courtes
        seq = seq + [vocab['<PAD>']] * (max_len - len(seq))
    else:
    # On coupe si c'est trop long
        seq = seq[:max_len]
    X_seq.append(seq)

# 5. Conversion en Tenseurs PyTorch et séparation Train/Test
X_tensor = torch.tensor(X_seq, dtype=torch.long)
y_tensor = torch.tensor(labels, dtype=torch.float32)

X_train, X_test, y_train, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)

# 6. Création des DataLoaders (pour traiter par lots/batches de 64)
batch_size = 64
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

print(f"Taille de l'entraînement : {len(X_train)} avis. Taille du test : {len(X_test)} avis.")

Chargement des données...
Mon dossier actuel est : /Users/lorispigneaux/Desktop/3ème année BUT info/Semestre 6/SAE6/PROJET_YELP_S6/notebooks
Chargement de 50000 lignes depuis ../data/raw/review.json...
Taille de l'entraînement : 40000 avis. Taille du test : 10000 avis.


### 2. Architecture du Modèle PyTorch (TextCNN)

In [3]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_filters, filter_sizes):
        super(TextCNN, self).__init__()
        # Couche d'Embedding : transforme les numéros de mots en vecteurs denses
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # Couches de Convolution 1D : glissent sur le texte par groupes de N mots
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])
        
        # Couche finale de classification
        self.fc = nn.Linear(len(filter_sizes) * num_filters, 1)
        self.dropout = nn.Dropout(0.5) # Régularisation
        self.sigmoid = nn.Sigmoid()    # Probabilité entre 0 et 1

    def forward(self, text):
        # Dimensions en entrée : [batch_size, seq_len]
        embedded = self.embedding(text) 
        # Dimensions après embedding : [batch_size, seq_len, embed_dim]
        
        # PyTorch Conv1d attend [batch_size, in_channels, seq_len]
        embedded = embedded.permute(0, 2, 1) 
        
        # Convolutions + Fonction d'activation ReLU + Max Pooling sur toute la longueur
        conved = [torch.relu(conv(embedded)) for conv in self.convs]
        pooled = [torch.max(conv, dim=2)[0] for conv in conved]
        
        # Concaténation des caractéristiques extraites par les différents filtres
        cat = self.dropout(torch.cat(pooled, dim=1))
        
        return self.sigmoid(self.fc(cat)).squeeze()

### 3. Entraînement du Modèle

In [4]:
# --- Configuration du modèle ---
embed_dim = 100        # Taille des vecteurs de mots
num_filters = 100      # Nombre de filtres par taille
filter_sizes = [2, 3, 4] # Recherche d'expressions de 2, 3 et 4 mots
num_epochs = 5         # Nombre de passages sur les données
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Appareil utilisé pour l'entraînement : {device}")

model = TextCNN(vocab_size, embed_dim, num_filters, filter_sizes).to(device)
criterion = nn.BCELoss() # Binary Cross Entropy (classification binaire)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# --- Boucle d'entraînement ---
print("\nDébut de l'entraînement...")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()            # 1. Remise à zéro des gradients
        predictions = model(batch_X)     # 2. Prédictions (Forward)
        loss = criterion(predictions, batch_y) # 3. Calcul de la perte
        loss.backward()                  # 4. Rétropropagation (Backward)
        optimizer.step()                 # 5. Mise à jour des poids
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{num_epochs} | Loss moyenne : {total_loss/len(train_loader):.4f}")

Appareil utilisé pour l'entraînement : cpu

Début de l'entraînement...
Epoch 1/5 | Loss moyenne : 0.4156
Epoch 2/5 | Loss moyenne : 0.3013
Epoch 3/5 | Loss moyenne : 0.2508
Epoch 4/5 | Loss moyenne : 0.2127
Epoch 5/5 | Loss moyenne : 0.1891


### 4. Évaluation des Performances

In [5]:
model.eval()
all_preds = []
all_targets = []

print("Évaluation sur le jeu de test...")
with torch.no_grad(): # Désactivation des gradients (plus rapide, moins de mémoire)
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        predictions = model(batch_X)
        
        # Si probabilité > 0.5, on classe comme Négatif (1), sinon Positif (0)
        rounded_preds = torch.round(predictions).cpu().numpy()
        all_preds.extend(rounded_preds)
        all_targets.extend(batch_y.numpy())

print("\nRapport de Classification du CNN PyTorch :\n")
print(classification_report(all_targets, all_preds, target_names=["Positif (>2 étoiles)", "Négatif (1-2 étoiles)"]))


Évaluation sur le jeu de test...

Rapport de Classification du CNN PyTorch :

                       precision    recall  f1-score   support

 Positif (>2 étoiles)       0.94      0.94      0.94      7710
Négatif (1-2 étoiles)       0.80      0.81      0.80      2290

             accuracy                           0.91     10000
            macro avg       0.87      0.87      0.87     10000
         weighted avg       0.91      0.91      0.91     10000

